# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not subsript or iterate over dataset.metadata)
metadata_json = dataset.metadata.to_json()
print("Dataset Name:", metadata_json["name"])
print("Description:", metadata_json["description"])

# Print licensing and temporal coverage
print("License:", metadata_json.get("license", "(unknown)"))
print("Temporal Coverage:", metadata_json.get("temporalCoverage", "(unknown)"))

## 2. Data Overview
Review the available record sets, fields, and their `@id` values.

*Each entity in the dataset is uniquely referenced by its `@id` field.*

Let's inspect the record sets in this dataset, and enumerate their fields.

In [ ]:
# Retrieve record sets
record_sets = dataset.metadata.record_sets()
print("Record Sets and their @id values:")
for rs in record_sets:
    print(f"  - name: {rs.name}")
    print(f"    @id: {rs.id}")
    print(f"    description: {rs.description}")
    # List fields in each record set
    print("    Fields:")
    for fld in rs.fields:
        print(f"      - {fld.name}    (@id: {fld.id}, dataType: {getattr(fld, 'data_type', '')})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Reference each record set and field by its `@id` as shown above.

In [ ]:
# Extract all record sets (@id references)
# Build a mapping of record set name to @id
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"[{rs.name}] DataFrame loaded. Columns (@id): {df.columns.tolist()}")
    print(df.head(2), "\n")
# Choose first record set for exploration
explore_rs_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, categorizing data, and grouping.

We reference fields and attributes with their `@id`. Choose a numeric field for this demonstration.

In [ ]:
# Identify and select a numeric field (@id) from the record set for analysis
eda_rs_id = explore_rs_id
if eda_rs_id:
    eda_rs = [rs for rs in record_sets if rs.id == eda_rs_id][0]
    # Find a numeric field
    numeric_field = None
    for field in eda_rs.fields:
        if (getattr(field, 'data_type', None) in ['schema:Float', 'schema:Integer', 'Float', 'Integer']):
            numeric_field = field.id
            numeric_field_name = field.name
            break
    print(f"Numeric field selected for EDA: {numeric_field} ({numeric_field_name})")
else:
    print("No record set found.")

if numeric_field and eda_rs_id:
    df = dataframes[eda_rs_id]
    # Thresholding: filter records for numeric_field > threshold
    threshold = 10
    # Coerce numeric field to numeric type safely
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} (@id: {numeric_field}) > {threshold}:")
    print(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Find a field suitable for grouping (categorical or string type)
    group_field = None
    for field in eda_rs.fields:
        if getattr(field, 'data_type', None) in ['schema:Text', 'Text'] and field.id != numeric_field:
            group_field = field.id
            group_field_name = field.name
            break
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} ({group_field_name}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields.

*Bar plot and histogram for filtered numeric field; boxplot for grouped data.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and eda_rs_id:
    # Histogram for numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[numeric_field], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_name} (@id: {numeric_field}) filtered > {threshold}")
    plt.xlabel(numeric_field_name)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by group_field if applicable
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field_name} grouped by {group_field_name}")
        plt.xticks(rotation=45)
        plt.show()

else:
    print("No numeric field or suitable grouping for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library for FAIR^2-compliant dataset exploration:

- Dataset metadata parsing and description inspection.
- Record sets, fields, and their `@id` references for robust data handling.
- Extraction to pandas DataFrames for each record set.
- Numeric and categorical field EDA, including filtering and normalization.
- Visualization of the distribution and grouped characteristics.

Further analysis can be tailored to clinical research questions, leveraging the reliability of the Croissant schema and MLCommons tools.

(End of notebook)